In [ ]:
# Importing libraries and defining paths

from ultralytics import YOLO
from object_detection.config.loader import PROJECT_ROOT, load_config

cfg = load_config()
print(cfg.data.detector_yaml)

In [ ]:
# Training the baseline model

model = YOLO(cfg.model.weights)

results = model.train(
    data=str(cfg.data.detector_yaml),
    epochs=cfg.train.epochs,
    batch=cfg.train.batch,
    imgsz=cfg.train.imgsz,
    seed=cfg.train.seed,
    project=str(PROJECT_ROOT / "runs"),
    name="baseline",
)

In [ ]:
# Loading the best weights

BEST_WEIGHTS = cfg.inference.weights
best = YOLO(str(BEST_WEIGHTS))

In [ ]:
# Inspecting one prediction

VALID = cfg.data.detector_yaml.parent / "valid"
val_imgs = sorted((VALID / "images").glob("*"))

r = best(str(val_imgs[0]), conf=cfg.inference.conf, verbose=False)[0]
print("image (height, width):", r.orig_shape)

for xyxy, conf in zip(r.boxes.xyxy.cpu().tolist(), r.boxes.conf.cpu().tolist()):
    print([round(v) for v in xyxy], round(conf, 3))

In [ ]:
# Visualising predictions

from IPython.display import display
from PIL import Image

for img_path in val_imgs[:15]:
    r = best(str(img_path), conf=cfg.inference.conf, verbose=False)[0]
    print(img_path.name, "-", len(r.boxes), "detections")

    annotated = Image.fromarray(r.plot()[:, :, ::-1])
    annotated.thumbnail((800, 800))
    display(annotated)


First time running the model, some images have more boxes than there are objects. This may have multiple factors, including but not limited to diverse object selection (with some having differing shapes depending on state), not enough training data, and many more.

In [ ]:
# Background check: images with no objects should get no boxes

for img_path in val_imgs:
    label_path = VALID / "labels" / (img_path.stem + ".txt")
    is_background = not label_path.exists() or label_path.read_text().strip() == ""

    if is_background:
        r = best(str(img_path), conf=cfg.inference.conf, verbose=False)[0]
        confidences = [round(c, 3) for c in r.boxes.conf.cpu().tolist()]
        print(img_path.name, "detections:", len(r.boxes), confidences)

The background check shows that one of the background images have 2 predicted boxes. Since all background-only images are verified to contain no objects, this is a genuine false positive. There are a few strategies to mitigate this, including but not limited to having more training data, or raising ```conf``` to ~0.62, but this will also make the model not detect some of the lower-confidence objects which will raise the amount of false negatives. This is also one trade-off to consider and measure.

Looking at the actual image and the predictions, both predictions are for the handles of the chair which is one of the backgrounds. Perhaps it is viable to include more images of this particular background with the handles visible to maybe reduce these types of errors.